# YOLOv8：图片、标注与目标检测

通过 COCO128 认识图片、标注和数据集配置。

## 1. 图片和标注

每张图片在 `images/train2017/`，同名的 `.txt` 标注放在 `labels/train2017/`。一行表示一个目标：`类别编号 中心点x 中心点y 宽度 高度`。后四个数相对于图片宽高，数值通常在 0 到 1 之间。没有目标的图片可以没有标注文件。

In [ ]:
from pathlib import Path
import yaml
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

project = Path.cwd().resolve()
if project.name == "notebooks":
    project = project.parent
dataset = project / "datasets" / "coco128"
images = sorted((dataset / "images" / "train2017").glob("*.jpg"))
labels = dataset / "labels" / "train2017"
config = yaml.safe_load((dataset / "coco128.yaml").read_text(encoding="utf-8"))
names = config["names"]
print(f"图片：{len(images)} 张，标注文件：{len(list(labels.glob('*.txt')))} 个")

## 2. 可视化真实标注

下面展示前 6 张图片及其真实标注，并保存结果。类别编号通过 `names` 转成类别名称。

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, image_path in zip(axes.flat, images[:6]):
    image = plt.imread(image_path)
    height, width = image.shape[:2]
    ax.imshow(image)
    label_path = labels / (image_path.stem + ".txt")
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            class_id, x, y, w, h = map(float, line.split())
            left, top = (x - w / 2) * width, (y - h / 2) * height
            ax.add_patch(Rectangle((left, top), w * width, h * height, fill=False, edgecolor="lime", linewidth=1.5))
            ax.text(left, top, names[int(class_id)], color="white", fontsize=8, bbox=dict(facecolor="green", alpha=0.7, pad=1))
    ax.set_title(image_path.name)
    ax.axis("off")
fig.tight_layout()
output = project / ".runtime" / "coco128_ground_truth.png"
output.parent.mkdir(exist_ok=True)
fig.savefig(output, dpi=150)
print("图片已保存到：", output)
plt.show()

## 3. 数据集 YAML

YAML 文件告诉 YOLO 训练图片、验证图片的位置，以及类别编号和名称。`scripts/dataset.py` 在运行时结合当前项目目录生成可用的本机路径。

**注意：**COCO128 的训练集和验证集使用相同图片，分数不适合衡量真实新场景的泛化能力。小火龙数据也很少，应该用独立的新照片检查效果。

In [ ]:
print((dataset / "coco128.yaml").read_text(encoding="utf-8")[:500])